## Xarray engine: temporal dimensions - seasonal forecast

In [1]:
import earthkit.data as ekd

ds_fl = ekd.from_source("sample", "seasonal_monthly.grib").to_fieldlist()

The input data contains seasonal monthly forecast. Because the length of a month varies, for this data the ``forecastMonth`` key is better suited for describing the temporal structure than using the ``step*`` keys. 

This is how the first few GRIB messages look like:

In [2]:
ds_fl[0:4].ls(extra_keys="metadata.forecastMonth")

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type,metadata.forecastMonth
0,2t,1993-11-01,1993-10-01,31 days,0,surface,0,regular_ll,1
1,2t,1993-11-01,1993-10-01,31 days,0,surface,1,regular_ll,1
2,2t,1993-11-01,1993-10-01,31 days,0,surface,2,regular_ll,1
3,2t,1993-12-01,1993-10-01,61 days,0,surface,0,regular_ll,2


In [3]:
ds = ds_fl.to_xarray(time_dim_mode="forecast", dim_roles={"step": "metadata.forecastMonth"})
ds

<xarray.Dataset> Size: 395kB
Dimensions:                  (member: 3, forecast_reference_time: 4, step: 6,
                              latitude: 19, longitude: 36)
Coordinates:
  * member                   (member) <U1 12B '0' '1' '2'
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 199...
  * step                     (step) int64 48B 1 2 3 4 5 6
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    2t                       (member, forecast_reference_time, step, latitude, longitude) float64 394kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

When we check the "step" dimension we can see its units are "months".

In [4]:
print(ds["step"])

<xarray.DataArray 'step' (step: 6)> Size: 48B
array([1, 2, 3, 4, 5, 6])
Coordinates:
  * step     (step) int64 48B 1 2 3 4 5 6
Attributes:
    units:    months


In [5]:
ds = ds_fl.to_xarray(
    time_dim_mode="forecast", dim_roles={"step": "metadata.forecastMonth"}, dim_name_from_role_name=False
)
ds

<xarray.Dataset> Size: 395kB
Dimensions:                  (member: 3, forecast_reference_time: 4,
                              forecastMonth: 6, latitude: 19, longitude: 36)
Coordinates:
  * member                   (member) <U1 12B '0' '1' '2'
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 199...
  * forecastMonth            (forecastMonth) int64 48B 1 2 3 4 5 6
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    2t                       (member, forecast_reference_time, forecastMonth, latitude, longitude) float64 394kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF